# Student Academic Performance Database

Dataset:
https://www.kaggle.com/datasets/therohithanand/student-academic-performance-dataset

In [24]:
from google.colab import userdata
import os

# Retrieve secret from Colab Secrets
token = userdata.get("KAGGLE_API_TOKEN")

# Set environment variable for Kaggle
os.environ["KAGGLE_API_TOKEN"] = token

NEON_CON = userdata.get("NEON_ADMIN")

In [25]:
import psycopg2
import pandas as pd


In [26]:
conn = psycopg2.connect(NEON_CON)
conn.autocommit = True

cur = conn.cursor()

try:
    cur.execute("CREATE DATABASE student_set_1;")
    print("Database student_set_1 created.")
except psycopg2.errors.DuplicateDatabase:
    print("Database student_set_1 already exists.")

cur.close()
conn.close()

Database student_set_1 already exists.


In [27]:
# Connect to an existing database (NOT the one you want to create)
conn = psycopg2.connect(
    NEON_CON
)
# VERY IMPORTANT
conn.autocommit = True

cur = conn.cursor()
#cur.execute("DROP DATABASE titanic_1;")
cur.close()
conn.close()

# Load Data From Kaggle

In [28]:
!pip -q uninstall -y kaggle kagglesdk
!pip -q install -U kagglesdk kaggle

In [29]:
!kaggle datasets download -d therohithanand/student-academic-performance-dataset
!unzip -o student-academic-performance-dataset.zip

Dataset URL: https://www.kaggle.com/datasets/therohithanand/student-academic-performance-dataset
License(s): CC-BY-SA-4.0
student-academic-performance-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  student-academic-performance-dataset.zip
  inflating: student_info.csv        


In [30]:
import pandas as pd
import glob

csv_files = glob.glob("*.csv")
print(csv_files)

df_students = pd.read_csv(csv_files[0])
df_students.head()

['student_info.csv']


,student_id,name,gender,age,grade_level,math_score,reading_score,writing_score,attendance_rate,parent_education,study_hours,internet_access,lunch_type,extra_activities,final_result
0,S1,Student_1,Other,17,10,74,61,90,94.660002,Master's,4.120192,Yes,Free or reduced,Yes,Fail
1,S2,Student_2,Male,17,12,99,70,91,93.173227,Bachelor's,2.886505,No,Free or reduced,No,Pass
2,S3,Student_3,Other,17,9,59,60,99,98.631098,PhD,1.909926,No,Free or reduced,No,Fail
3,S4,Student_4,Other,17,12,70,88,69,96.419620,PhD,1.664740,No,Standard,No,Pass
4,S5,Student_5,Male,15,9,85,77,94,91.332105,PhD,2.330918,Yes,Free or reduced,No,Pass


In [31]:
df_students.columns = df_students.columns.str.strip()
df_students.columns

Index(['student_id', 'name', 'gender', 'age', 'grade_level', 'math_score',
       'reading_score', 'writing_score', 'attendance_rate', 'parent_education',
       'study_hours', 'internet_access', 'lunch_type', 'extra_activities',
       'final_result'],
      dtype='object')

# Create Table

In [32]:
from urllib.parse import urlparse, urlunparse

def switch_database(connection_string, database_name):
    parsed = urlparse(connection_string)
    return urlunparse(parsed._replace(path=f"/{database_name}"))

NEON_STUDENTS = switch_database(NEON_CON, "student_set_1")

In [33]:
conn = psycopg2.connect(NEON_STUDENTS)
conn.autocommit = True

cur = conn.cursor()

create_tables_sql = """
DROP VIEW IF EXISTS student_academic_view;
DROP TABLE IF EXISTS academic_records;
DROP TABLE IF EXISTS student_profiles;

CREATE TABLE student_profiles (
    student_id TEXT PRIMARY KEY,
    student_name TEXT,
    gender TEXT,
    age INT,
    grade_level INT,
    parental_education TEXT,
    internet_access TEXT,
    lunch_type TEXT
);

CREATE TABLE academic_records (
    record_id SERIAL PRIMARY KEY,
    student_id TEXT REFERENCES student_profiles(student_id),
    math_score INT,
    reading_score INT,
    writing_score INT,
    attendance_rate FLOAT,
    study_time_weekly FLOAT,
    extracurricular TEXT,
    final_result TEXT
);
"""

cur.execute(create_tables_sql)

cur.close()
conn.close()

print("Tables created.")

Tables created.


In [34]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(NEON_STUDENTS)

# Rename Kaggle columns to PostgreSQL-friendly names
df_students = df_students.rename(columns={
    "name": "student_name",
    "parent_education": "parental_education",
    "study_hours": "study_time_weekly",
    "extra_activities": "extracurricular"
})

# Keep student_id as text, such as S1, S2, S3
df_students["student_id"] = df_students["student_id"].astype(str)

student_profiles = df_students[
    [
        "student_id",
        "student_name",
        "gender",
        "age",
        "grade_level",
        "parental_education",
        "internet_access",
        "lunch_type"
    ]
]

academic_records = df_students[
    [
        "student_id",
        "math_score",
        "reading_score",
        "writing_score",
        "attendance_rate",
        "study_time_weekly",
        "extracurricular",
        "final_result"
    ]
]

student_profiles.to_sql(
    "student_profiles",
    engine,
    if_exists="append",
    index=False,
    method="multi"
)

academic_records.to_sql(
    "academic_records",
    engine,
    if_exists="append",
    index=False,
    method="multi"
)

print("Full student dataset inserted into Neon.")

Full student dataset inserted into Neon.


In [35]:
df_check_profiles = pd.read_sql_query(
    "SELECT COUNT(*) AS total_student_profiles FROM student_profiles;",
    engine
)

df_check_records = pd.read_sql_query(
    "SELECT COUNT(*) AS total_academic_records FROM academic_records;",
    engine
)

display(df_check_profiles)
display(df_check_records)

,total_student_profiles
0,1000


,total_academic_records
0,1000


In [36]:
join_query = """
SELECT
    s.student_id,
    s.student_name,
    s.gender,
    s.age,
    s.grade_level,
    s.parental_education,
    s.internet_access,
    s.lunch_type,
    a.math_score,
    a.reading_score,
    a.writing_score,
    a.attendance_rate,
    a.study_time_weekly,
    a.extracurricular,
    a.final_result
FROM student_profiles s
JOIN academic_records a
    ON s.student_id = a.student_id
ORDER BY s.student_id
LIMIT 10;
"""

df_query = pd.read_sql_query(join_query, engine)
df_query

,student_id,student_name,gender,age,grade_level,parental_education,internet_access,lunch_type,math_score,reading_score,writing_score,attendance_rate,study_time_weekly,extracurricular,final_result
0,S1,Student_1,Other,17,10,Master's,Yes,Free or reduced,74,61,90,94.660002,4.120192,Yes,Fail
1,S10,Student_10,Other,17,11,Bachelor's,Yes,Standard,62,95,96,80.216222,1.716656,Yes,Fail
2,S100,Student_100,Male,17,10,High School,No,Free or reduced,93,57,94,83.136254,2.727228,No,Pass
3,S1000,Student_1000,Male,17,9,PhD,No,Free or reduced,96,92,93,84.695088,1.262504,Yes,Pass
4,S101,Student_101,Other,16,10,PhD,Yes,Free or reduced,54,54,96,96.546043,4.127119,Yes,Fail
5,S102,Student_102,Other,17,12,Bachelor's,Yes,Free or reduced,52,58,69,80.838304,2.936692,Yes,Fail
6,S103,Student_103,Other,15,9,Bachelor's,No,Free or reduced,50,50,68,88.383658,4.176058,No,Pass
7,S104,Student_104,Male,16,11,PhD,No,Free or reduced,53,85,84,83.364073,3.305956,Yes,Pass
8,S105,Student_105,Other,15,10,Master's,No,Free or reduced,89,84,53,97.586604,2.368278,Yes,Fail
9,S106,Student_106,Other,17,12,PhD,Yes,Free or reduced,84,51,54,91.140810,4.239915,No,Fail


In [37]:
report_query = """
SELECT
    s.parental_education,
    COUNT(*) AS number_of_students
FROM student_profiles s
JOIN academic_records a
    ON s.student_id = a.student_id
GROUP BY s.parental_education;
"""

df_report = pd.read_sql_query(report_query, engine)
df_report

,parental_education,number_of_students
0,PhD,254
1,Master's,227
2,High School,245
3,Bachelor's,274


# Now create a public user

In [38]:
STUDENT_READER_PASSWORD = userdata.get("STUDENT_READER_PASSWORD")

In [39]:
conn = psycopg2.connect(NEON_STUDENTS)
conn.autocommit = True

cur = conn.cursor()

cur.execute(f"""
DO $$
BEGIN
    IF NOT EXISTS (
        SELECT FROM pg_catalog.pg_roles
        WHERE rolname = 'student_reader'
    ) THEN
        CREATE ROLE student_reader
        WITH LOGIN
        PASSWORD '{STUDENT_READER_PASSWORD}';
    END IF;
END
$$;
""")

cur.execute("""
GRANT CONNECT ON DATABASE student_set_1 TO student_reader;
GRANT USAGE ON SCHEMA public TO student_reader;
GRANT SELECT ON student_profiles TO student_reader;
GRANT SELECT ON academic_records TO student_reader;
""")

cur.close()
conn.close()